In [1]:
#models
library(tidyverse)
library(caret)
library(recipes)
library(pROC)
library(dplyr)
library(tidyr)
library(ppcor)

dir.create("../../outputs/Extended/Frag-Boxplots", recursive = TRUE, showWarnings = FALSE)


── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: lattice


Attaching package: ‘caret’


The following object is masked from ‘package:purrr’:

    lift



Attaching package: ‘recipes’


The following object is masked from ‘package:stringr’:

    fixed


The following object is masked from ‘package:stats’:

    step


Type 'citation("pROC")' for a citation.


Attaching package: ‘pRO

In [2]:
mica_df <- read_csv("../../data/mica_df_with_pcs.csv")
lucas_df <- read_csv("../../data/lucas_df_with_pcs.csv")
val_df <- read_csv("../../data/val_df_with_pcs.csv")

mica_df <- mica_df %>%
  mutate(olink_CEA = log2(clinical_CEA + 1))

lucas_df <- lucas_df %>%
  mutate(olink_CEA = log2(clinical_CEA + 1))

mica_df_filtered <- mica_df[mica_df$`Coded Type` %in% c("Lung (LU)", "No cancer (NC)"), ]

Rows: 540 Columns: 1304
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr   (73): id, Stage, Coded Type, type, Delfi ID (Aliquot 2), DL ID, DL Suf...
dbl (1225): clinical_CCI, clinical_age, Smoking, ratio_1, ratio_2, ratio_3, ...
lgl    (6): Match?, Rack (plasma), Rack (cfDNA), check, Robot Protocol, Repl...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 287 Columns: 958
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr   (96): Patient, id, type, clinical_smokingstatus, QC, patient.type, cli...
dbl  (850): multinucratio, clinical_nlratio, clinical_CRP, clinical_cfdna_co...
lgl    (3): DIAGNOSE_OTHER_PRIMARY, Adaptor Dimer, Date Data Received
date   (9): DATE_1_V

## Plotting Fragmentation Signals by Stage

In [3]:
library(tidyverse)
library(ggpubr)
library(ggsignif)
library(patchwork)

plot_biomarker_stage <- function(
  df,
  biomarker_col  = "ratio_pc_08",
  biomarker_name = "PC8 (ratio_pc_08)",
  stage_col      = "Stage",
  ref_group      = "Non-cancer",
  test_method    = c("t.test", "wilcox.test", "anova", "kruskal.test"),
  color_scheme   = c("blue", "red", "gold", "green", "purple"),
  y_limits       = NULL,
  plot_title     = NULL,
  hide_outliers  = TRUE,
  show_y_axis    = TRUE
) {

  test_method  <- match.arg(test_method)
  color_scheme <- match.arg(color_scheme)

  # ---- Color schemes ----
  fill_colors <- switch(color_scheme,
    "blue"   = c("Non-cancer" = "grey50", "I/II" = "#9ECAE1", "III/IV" = "#08519C"),
    "red"    = c("Non-cancer" = "grey50", "I/II" = "#FC9272", "III/IV" = "#CB181D"),
    "gold"   = c("Non-cancer" = "grey50", "I/II" = "#FDD49E", "III/IV" = "#D95F0E"),
    "green"  = c("Non-cancer" = "grey50", "I/II" = "#A1D99B", "III/IV" = "#238B45"),
    "purple" = c("Non-cancer" = "grey50", "I/II" = "#BCBDDC", "III/IV" = "#6A51A3")
  )

  # ---- Data prep ----
  df <- df %>%
    filter(!is.na(.data[[biomarker_col]])) %>%
    mutate(
      stage_chr = as.character(.data[[stage_col]]),
      StageGroup = case_when(
        is.na(.data[[stage_col]]) |
          stage_chr %in% c("NA", "N/A", "Non-cancer",
                           "No baseline cancer", "Healthy", "Control") ~ "Non-cancer",
        grepl("^I($|[AB]|/)",  stage_chr, ignore.case = TRUE) |
          grepl("^II($|[AB]|/)", stage_chr, ignore.case = TRUE) ~ "I/II",
        grepl("^III", stage_chr, ignore.case = TRUE) |
          grepl("^IV",  stage_chr, ignore.case = TRUE) ~ "III/IV",
        TRUE ~ NA_character_
      ),
      StageGroup = factor(StageGroup, levels = c("Non-cancer", "I/II", "III/IV"))
    ) %>%
    filter(!is.na(StageGroup))

  if (nrow(df) == 0) stop("No valid rows after stage grouping/filtering.")

  # ---- Sample sizes (all data) ----
  sample_sizes <- df %>%
    count(StageGroup, name = "n") %>%
    arrange(StageGroup)

  # ---- Flag outliers (but keep them in the data) ----
  if (hide_outliers) {
    df <- df %>%
      group_by(StageGroup) %>%
      mutate(
        Q1 = quantile(.data[[biomarker_col]], 0.25, na.rm = TRUE),
        Q3 = quantile(.data[[biomarker_col]], 0.75, na.rm = TRUE),
        IQR = Q3 - Q1,
        lower_bound = Q1 - 1.5 * IQR,
        upper_bound = Q3 + 1.5 * IQR,
        is_outlier = .data[[biomarker_col]] < lower_bound |
          .data[[biomarker_col]] > upper_bound
      ) %>%
      dplyr::select(-Q1, -Q3, -IQR, -lower_bound, -upper_bound) %>%
      ungroup()
  } else {
    df$is_outlier <- FALSE
  }

  # ---- Comparisons ----
  levs_present <- levels(df$StageGroup)
  comparisons <- if (ref_group %in% levs_present) {
    lapply(setdiff(levs_present, ref_group), function(x) c(ref_group, x))
  } else if (length(levs_present) >= 2) {
    combn(levs_present, 2, simplify = FALSE)
  } else {
    list()
  }

  # ---- Compute y-limits for DISPLAY (zoom) using non-outliers if requested ----
  plot_values <- if (hide_outliers) {
    df %>% filter(!is_outlier) %>% pull(.data[[biomarker_col]])
  } else {
    df[[biomarker_col]]
  }
  plot_values <- plot_values[is.finite(plot_values)]

  if (length(plot_values) == 0) stop("No finite biomarker values available to plot.")

  y_min_local <- min(plot_values, na.rm = TRUE)
  y_max_local <- max(plot_values, na.rm = TRUE)
  y_rng_local <- y_max_local - y_min_local
  if (!is.finite(y_rng_local) || y_rng_local <= 0) y_rng_local <- 1

  if (is.null(y_limits)) {
    y_limits <- c(
      y_min_local - 0.10 * y_rng_local,
      y_max_local + 0.22 * y_rng_local    # increased from 0.15
    )
  }

  # ---- Robust positioning for p-value brackets and n labels ----
  y_range_for_positioning <- y_limits[2] - y_limits[1]
  top <- y_limits[2]
  step <- 0.09 * y_range_for_positioning  # increased from 0.05

  label_y <- if (length(comparisons) > 0) {
    yy <- top - seq(from = step, by = step, length.out = length(comparisons))
    rev(yy)
  } else {
    NULL
  }

  n_y <- y_limits[1] + 0.03 * y_range_for_positioning

  # ---- Base plot ----
  p <- ggplot(df, aes(x = StageGroup, y = .data[[biomarker_col]], fill = StageGroup)) +
    geom_boxplot(alpha = 0.7, outlier.shape = NA, width = 0.5) +
    geom_jitter(
      data = df %>% filter(!is_outlier),
      aes(color = StageGroup),
      width = 0.15, size = 4, alpha = 0.6
    ) +
    scale_fill_manual(values = fill_colors, guide = "none") +
    scale_color_manual(values = fill_colors, guide = "none") +
    labs(
      x = NULL,
      y = if (show_y_axis) biomarker_name else NULL,
      title = plot_title
    ) +
    scale_y_continuous(
      expand = expansion(mult = c(0.02, 0.08)),
      breaks = scales::pretty_breaks(n = 6)
    ) +
    coord_cartesian(ylim = y_limits, clip = "off") +
    theme_classic(base_size = 20) +
    theme(
      axis.title.y = element_text(size = 42, margin = margin(r = 20)),
      axis.text.x  = element_text(size = 36, color = "black", margin = margin(t = 10)),
      axis.text.y  = if (show_y_axis) element_text(size = 32) else element_blank(),
      axis.ticks.y = if (show_y_axis) element_line() else element_blank(),
      axis.line.y  = if (show_y_axis) element_line(color = "black", linewidth = 1) else element_blank(),
      panel.grid.major.y = element_line(color = "grey90", size = 0.3),
      panel.grid.minor   = element_blank(),
      axis.line = element_line(color = "black", linewidth = 1),
      legend.position = "none",
      plot.margin = margin(12, 22, 20, 10),
      plot.title = element_text(size = 36, hjust = 0.5)
    )

  # ---- Statistical tests (uses ALL data including outliers) ----
  # NOTE: ggpubr::stat_compare_means() silently IGNORES p.adjust.method when
  # `comparisons` is supplied (it delegates to ggsignif::geom_signif, which
  # does not adjust). To guarantee the stars reflect BH-adjusted p-values,
  # p-values are computed and adjusted manually here, then drawn directly
  # with ggsignif::geom_signif() using the precomputed labels.
  pval_table <- NULL

  if (test_method %in% c("t.test", "wilcox.test")) {
    if (length(comparisons) > 0) {
      test_fun <- if (test_method == "t.test") t.test else wilcox.test
      group_vals <- split(df[[biomarker_col]], df$StageGroup)

      raw_pvals <- vapply(comparisons, function(cmp) {
        x <- group_vals[[cmp[1]]]
        y <- group_vals[[cmp[2]]]
        tryCatch(test_fun(x, y)$p.value, error = function(e) NA_real_)
      }, numeric(1))

      adj_pvals <- p.adjust(raw_pvals, method = "BH")

      comparison_labels <- vapply(comparisons, function(cmp) paste(cmp, collapse = " vs "), character(1))
      pval_table <- data.frame(
        comparison = comparison_labels,
        p_raw      = signif(raw_pvals, 4),
        p_BH       = signif(adj_pvals, 4)
      )
      message(sprintf("\n[%s] %s (%s):", biomarker_col, if (!is.null(plot_title)) plot_title else "", test_method))
      message(paste(capture.output(print(pval_table, row.names = FALSE)), collapse = "\n"))

      sig_stars <- symnum(
        adj_pvals, corr = FALSE, na = FALSE,
        cutpoints = c(0, 0.0001, 0.001, 0.01, 0.05, 1),
        symbols   = c("****", "***", "**", "*", "ns")
      )

      p <- p + ggsignif::geom_signif(
        comparisons = comparisons,
        annotations = as.character(sig_stars),
        y_position  = label_y,
        tip_length  = 0.001,
        size        = 1.2,
        textsize    = 14
      )
    }
  } else if (test_method %in% c("anova", "kruskal.test")) {
    p <- p + stat_compare_means(
      method   = test_method,
      label    = "p.format",
      label.y  = y_limits[2] - 0.08 * y_range_for_positioning,
      size     = 14
    )
  }

  # ---- "n =" labels ----
  for (i in seq_len(nrow(sample_sizes))) {
    p <- p + annotate(
      "text",
      x     = i,
      y     = n_y,
      label = paste0("n = ", sample_sizes$n[i]),
      size  = 10,
      color = "black"
    )
  }

  attr(p, "pval_table") <- pval_table
  p
}

plot_biomarker_stage_multi <- function(
  dfs,
  biomarker_col,
  biomarker_name,
  stage_col     = "Stage",
  ref_group     = "Non-cancer",
  test_method   = "t.test",
  color_schemes = NULL,
  plot_titles   = NULL,
  ncol          = 2,
  hide_outliers = TRUE
) {
  if (is.null(color_schemes)) {
    base_schemes <- c("blue", "green", "red", "gold", "purple")
    color_schemes <- rep(base_schemes, length.out = length(dfs))
  }
  if (length(color_schemes) != length(dfs)) stop("`color_schemes` must have same length as `dfs`.")

  if (is.null(plot_titles)) plot_titles <- rep(list(NULL), length(dfs))
  if (length(plot_titles) != length(dfs)) stop("`plot_titles` must have same length as `dfs`.")

  # ---- Prepare dfs and flag outliers ----
  processed_dfs <- lapply(dfs, function(df) {
    df_proc <- df %>%
      filter(!is.na(.data[[biomarker_col]])) %>%
      mutate(
        stage_chr = as.character(.data[[stage_col]]),
        StageGroup = case_when(
          is.na(.data[[stage_col]]) |
            stage_chr %in% c("NA", "N/A", "Non-cancer",
                             "No baseline cancer", "Healthy", "Control") ~ "Non-cancer",
          grepl("^I($|[AB]|/)",  stage_chr, ignore.case = TRUE) |
            grepl("^II($|[AB]|/)", stage_chr, ignore.case = TRUE) ~ "I/II",
          grepl("^III", stage_chr, ignore.case = TRUE) |
            grepl("^IV",  stage_chr, ignore.case = TRUE) ~ "III/IV",
          TRUE ~ NA_character_
        ),
        StageGroup = factor(StageGroup, levels = c("Non-cancer", "I/II", "III/IV"))
      ) %>%
      filter(!is.na(StageGroup))

    if (hide_outliers) {
      df_proc <- df_proc %>%
        group_by(StageGroup) %>%
        mutate(
          Q1 = quantile(.data[[biomarker_col]], 0.25, na.rm = TRUE),
          Q3 = quantile(.data[[biomarker_col]], 0.75, na.rm = TRUE),
          IQR = Q3 - Q1,
          lower_bound = Q1 - 1.5 * IQR,
          upper_bound = Q3 + 1.5 * IQR,
          is_outlier = .data[[biomarker_col]] < lower_bound |
            .data[[biomarker_col]] > upper_bound
        ) %>%
        dplyr::select(-Q1, -Q3, -IQR, -lower_bound, -upper_bound) %>%
        ungroup()
    } else {
      df_proc$is_outlier <- FALSE
    }

    df_proc
  })

  # ---- Compute GLOBAL y-limits ----
  all_values <- unlist(lapply(processed_dfs, function(d) {
    vals <- if (hide_outliers) {
      d %>% filter(!is_outlier) %>% pull(.data[[biomarker_col]])
    } else {
      d[[biomarker_col]]
    }
    vals[is.finite(vals)]
  }))

  if (length(all_values) == 0) stop("No finite biomarker values found across all dfs.")

  global_min <- min(all_values, na.rm = TRUE)
  global_max <- max(all_values, na.rm = TRUE)
  global_rng <- global_max - global_min
  if (!is.finite(global_rng) || global_rng <= 0) global_rng <- 1

  y_limits <- c(
    global_min - 0.10 * global_rng,
    global_max + 0.22 * global_rng    # increased from 0.15
  )

  # ---- Build plots (only first plot shows y-axis) ----
  plot_list <- purrr::pmap(
    list(dfs, color_schemes, plot_titles, seq_along(dfs)),
    function(df, scheme, title, idx) {
      plot_biomarker_stage(
        df             = df,
        biomarker_col  = biomarker_col,
        biomarker_name = biomarker_name,
        stage_col      = stage_col,
        ref_group      = ref_group,
        test_method    = test_method,
        color_scheme   = scheme,
        y_limits       = y_limits,
        plot_title     = title,
        hide_outliers  = hide_outliers,
        show_y_axis    = (idx == 1)
      )
    }
  )

  wrap_plots(plotlist = plot_list, ncol = ncol)
}

# Usage example
options(repr.plot.width = 16, repr.plot.height = 16, repr.plot.res = 600)

p <- plot_biomarker_stage_multi(
  dfs = list(lucas_df, val_df),
  biomarker_col  = "ratio_pc_04",
  biomarker_name = "Fragmentome PC4",
  stage_col      = "Stage",
  ref_group      = "Non-cancer",
  test_method    = "wilcox.test",
  color_schemes  = c("blue", "purple"),
  plot_titles    = c("LUCAS Cohort\n", "JHU Cohort\n"),
  ncol           = 2,
  hide_outliers  = TRUE
)

print(p)


Attaching package: ‘patchwork’


The following object is masked from ‘package:MASS’:

    area


Warning message:
“The `size` argument of `element_line()` is deprecated as of ggplot2 3.4.0.
ℹ Please use the `linewidth` argument instead.”

[ratio_pc_04] LUCAS Cohort
 (wilcox.test):

           comparison     p_raw      p_BH
   Non-cancer vs I/II 2.251e-03 2.251e-03
 Non-cancer vs III/IV 4.728e-07 9.457e-07


[ratio_pc_04] JHU Cohort
 (wilcox.test):

           comparison     p_raw    p_BH
   Non-cancer vs I/II 7.998e-05 0.00016
 Non-cancer vs III/IV 2.886e-01 0.28860



In [4]:
save_all_ratio_pc_plots <- function(
  dfs,
  pc_range       = 1:11,
  output_dir     = "../../outputs/Fig5/Frag-Boxplots",
  stage_col      = "Stage",
  ref_group      = "Non-cancer",
  test_method    = "wilcox.test",
  color_schemes  = c("blue", "purple"),
  plot_titles    = c("LUCAS Cohort\n", "JHU Cohort\n"),
  ncol           = 2,
  hide_outliers  = TRUE,
  invert_pcs     = c(1, 2),      # PCs whose sign is flipped for display
  width          = 16,
  height         = 16,
  dpi            = 600
) {

  if (!dir.exists(output_dir)) {
    dir.create(output_dir, recursive = TRUE)
  }

  for (pc_num in pc_range) {

    biomarker_col <- sprintf("ratio_pc_%02d", pc_num)

    col_exists <- all(sapply(dfs, function(df) biomarker_col %in% names(df)))
    if (!col_exists) {
      message(sprintf("Skipping PC%d: column '%s' not found in all dataframes",
                      pc_num, biomarker_col))
      next
    }

    # ---- sign inversion for display -----------------------------------
    # PC loadings are sign-arbitrary; flipping selected PCs orients them so
    # higher = more cancer-like, consistent with the other panels. Negation is
    # monotone, so Wilcoxon p-values are unchanged.
    invert_this <- pc_num %in% invert_pcs

    dfs_use <- if (invert_this) {
      lapply(dfs, function(df) {
        df[[biomarker_col]] <- -df[[biomarker_col]]
        df
      })
    } else {
      dfs
    }

    biomarker_name <- if (invert_this) {
      sprintf("Fragmentome PC%d", pc_num)
    } else {
      sprintf("Fragmentome PC%d", pc_num)
    }

    filename <- file.path(
      output_dir,
      sprintf("Frag-PC%02d-Boxplot%s.png", pc_num, if (invert_this) "" else "")
    )

    message(sprintf("Creating plot for %s%s...",
                    biomarker_col, if (invert_this) "" else ""))

    p <- plot_biomarker_stage_multi(
      dfs            = dfs_use,
      biomarker_col  = biomarker_col,
      biomarker_name = biomarker_name,
      stage_col      = stage_col,
      ref_group      = ref_group,
      test_method    = test_method,
      color_schemes  = color_schemes,
      plot_titles    = plot_titles,
      ncol           = ncol,
      hide_outliers  = hide_outliers
    )

    ggsave(
      filename = filename,
      plot     = p,
      width    = width,
      height   = height,
      dpi      = dpi,
      units    = "in"
    )

    message(sprintf("Saved: %s", filename))
  }

  message("Done! All plots saved.")
}

# Usage:
save_all_ratio_pc_plots(
  dfs        = list(lucas_df, val_df),
  pc_range   = 1:11,
  output_dir = "../../outputs/Extended/Frag-Boxplots",
  invert_pcs = c(1, 2)
)

Creating plot for ratio_pc_01...


[ratio_pc_01] LUCAS Cohort
 (wilcox.test):

           comparison     p_raw      p_BH
   Non-cancer vs I/II 4.331e-01 4.331e-01
 Non-cancer vs III/IV 1.232e-10 2.464e-10


[ratio_pc_01] JHU Cohort
 (wilcox.test):

           comparison    p_raw    p_BH
   Non-cancer vs I/II 0.005785 0.01022
 Non-cancer vs III/IV 0.010220 0.01022

Saved: ../../outputs/Extended/Frag-Boxplots/Frag-PC01-Boxplot.png

Creating plot for ratio_pc_02...


[ratio_pc_02] LUCAS Cohort
 (wilcox.test):

           comparison     p_raw      p_BH
   Non-cancer vs I/II 4.539e-01 4.539e-01
 Non-cancer vs III/IV 1.156e-21 2.312e-21


[ratio_pc_02] JHU Cohort
 (wilcox.test):

           comparison     p_raw      p_BH
   Non-cancer vs I/II 0.2790000 0.2790000
 Non-cancer vs III/IV 0.0001659 0.0003319

Saved: ../../outputs/Extended/Frag-Boxplots/Frag-PC02-Boxplot.png

Creating plot for ratio_pc_03...


[ratio_pc_03] LUCAS Cohort
 (wilcox.test):

           comparison  p_raw   p_BH
   Non-c

In [ ]:
# Done #